In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Read the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Inspect the first few rows
df.head()

In [ ]:
# Task 3: Display dataset information
df.info()

In [ ]:
# Task 4: Show statistical description
df.describe()

In [ ]:
# Task 1: Handle missing values
print("Missing values before:")
print(df.isnull().sum())

# Fill missing values
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print("\nMissing values after:")
print(df.isnull().sum().sum())

In [ ]:
# Task 2: Check and remove duplicates
print(f"Duplicates before: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicates after: {df.duplicated().sum()}")

In [ ]:
# Task 3: Encode categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 4: Apply feature scaling (StandardScaler)
from sklearn.preprocessing import StandardScaler

target_col = 'Target'  # Correct column name

X = df.drop(target_col, axis=1)
y = df[target_col]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
# Task 5: Check for target imbalance
print("Target distribution:")
print(y.value_counts())
print(f"\nPercentage:\n{y.value_counts(normalize=True) * 100}")

# Plot target distribution
plt.figure(figsize=(6, 4))
y.value_counts().plot(kind='bar', edgecolor='black')
plt.title('Target Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

# Check imbalance ratio
imbalance_ratio = y.value_counts().min() / y.value_counts().max()
print(f"\nImbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio < 0.8:
    print("The dataset IS IMBALANCED - Use F1 Score and StratifiedKFold")
else:
    print("The dataset is balanced - Can use Accuracy")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2, 3, 4, 5: StratifiedKFold, CatBoost, F1 Score, Print averaged score
!pip install catboost -q

from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score

# Use StratifiedKFold for classification (maintains class distribution)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
accuracy_scores = []
model = None

for train_idx, val_idx in skfold.split(X_scaled, y):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train CatBoostClassifier
    model = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)

    # Calculate F1 Score (use for imbalanced data)
    f1 = f1_score(y_val, y_pred)
    acc = accuracy_score(y_val, y_pred)

    f1_scores.append(f1)
    accuracy_scores.append(acc)

# Print averaged scores
print(f"F1 Scores per fold: {f1_scores}")
print(f"Average F1 Score: {np.mean(f1_scores):.4f}")
print(f"\nAccuracy Scores per fold: {accuracy_scores}")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")

In [ ]:
# Task 1: Plot feature importance
feature_importance = model.feature_importances_
feature_names = X_scaled.columns

# Sort by importance
sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 20))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx])
plt.yticks(range(len(sorted_idx)), feature_names[sorted_idx], fontsize=6)
plt.xlabel('Feature Importance')
plt.title('CatBoost Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Identify the golden feature (most important)
golden_feature_idx = np.argmax(feature_importance)
golden_feature = feature_names[golden_feature_idx]

print(f"The GOLDEN FEATURE is: {golden_feature}")
print(f"Importance score: {feature_importance[golden_feature_idx]:.4f}")

In [ ]:
# Bonus: Retrain using ONLY the golden feature

# 1. Create new X with only the golden feature
X_golden = X_scaled[[golden_feature]]

# 2. Run StratifiedKFold with single feature
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

golden_f1_scores = []
golden_accuracy_scores = []

for train_idx, val_idx in skfold.split(X_golden, y):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_golden = CatBoostClassifier(iterations=100, random_state=42, verbose=0)
    model_golden.fit(X_train, y_train)

    y_pred = model_golden.predict(X_val)

    f1 = f1_score(y_val, y_pred)
    acc = accuracy_score(y_val, y_pred)

    golden_f1_scores.append(f1)
    golden_accuracy_scores.append(acc)

# 3. Print and compare
print("=" * 50)
print("COMPARISON: Full Model vs Golden Feature Only")
print("=" * 50)
print(f"\nFull Model Average F1 Score:    {np.mean(f1_scores):.4f}")
print(f"Golden Feature Average F1 Score: {np.mean(golden_f1_scores):.4f}")
print(f"\nFull Model Average Accuracy:    {np.mean(accuracy_scores):.4f}")
print(f"Golden Feature Average Accuracy: {np.mean(golden_accuracy_scores):.4f}")
print("=" * 50)